# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [140]:
!cat publications.xlsx

?�9L�ҙ�sbgٮ|�l!��USh9i�b�r:"y_dl��D���|-N��R"4�2�G�%��Z�4�˝y�7	ë��ɂ���  �� PK    ! /J�X�  F     xl/workbook.xml�U]o�0}�����N�h�J*��-�6U]?+�8�                                                                                                                                                                                                                                                                                                                                                                                                                     ���N�0E�H�C�-Jܲ5��*Q>�ēƪc[�ii����B�j7���{2��h�nm���ƻR����U^7/���%��rZY�@1__�f� �q��R4D�AJ�h>����V�ƹ�Z�9����NV�8ʩ����ji){^��-I�"{�v^�P!XS)bR�r��K�s(�3�`c�0��������7M4�����ZƐk+�|\|z�(���P��6h_-[�@�!��� Pk���2n�}�?�L��� ��%���d����dN"m,�ǞDO97*�~��ɸ8�O�c|n���E������B��!$}�����;{���[����2�  �� PK    ! �U0#�   L  _rels/.rels �(�                                                           

## Import pandas

We are using the very handy pandas library for dataframes.

In [141]:
import pandas as pd

## Import the Publication File (.tsv or .xlsx)

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [142]:
publications = pd.read_excel("publications.xlsx", header=0)
publications


,pub_date,type,title,venue,excerpt,citation,url_slug,paper_url,slides_url
0,2023-12-27,manuscripts,An exploration of robot-mediated Tai Chi exerc...,Applied Sciences,"In this fast-aging society, many older adults ...","Zheng, Z., Oh, H., Mim, M., Choi, W., & Lee, Y...",AS,https://www.mdpi.com/2076-3417/13/9/5306,https://www.mdpi.com/2076-3417/13/9/5306
1,2023-12-28,manuscripts,Developing a platform-specific framework for w...,Information Processing and Management,NaN,"Choi, W., Stvilia, B. & Lee, H. S. (2023). Dev...",IPM,NaN,NaN
2,2022-12-29,manuscripts,Design matters in web credibility assessment: ...,Asian Communication Research,NaN,"Choi, W., Kim, S.Y., & Luo, M. (2022). Design ...",ACR,NaN,NaN
3,2022-12-30,manuscripts,Understanding the research landscape of inform...,Gerontechnology,NaN,"Park, M., Lee, Y., & Choi, W. (2022). Understa...",Geron,NaN,NaN
4,2022-12-31,manuscripts,Respite service use among dementia and nondeme...,Journal of Applied Gerontology,NaN,"Lee, Y., Choi, W., & Park, M. (2022). Respite ...",JAG,NaN,NaN
5,2022-01-01,manuscripts,Associations between mastery of life and every...,Journal of the Association for Information Sci...,NaN,"Choi, W., Park, M. S., & Lee, Y. (2022). Assoc...",JASIST,NaN,NaN
6,2021-01-02,manuscripts,A feature-oriented analysis of developers’ des...,International Journal of Medical Informatics,NaN,"Wang, S., Lee, H. S., & Choi, W. (2021). A fea...",IJMI,NaN,NaN
7,2020-01-03,manuscripts,A systematic review of mobile health technolog...,Journal of the American Medical Informatics As...,NaN,"Choi, W., Wang, S., Lee, Y., Oh, H., & Zheng, ...",JAMIA,NaN,NaN
8,2019-01-04,manuscripts,Older adults’ credibility assessment of online...,Journal of the Association for Information Sci...,NaN,"Choi, W. (2020). Older adults’ credibility ass...",JASIST,NaN,NaN
9,2019-01-05,manuscripts,Older adults’ health information behavior in e...,Library & Information Science Research,NaN,"Choi, W. (2019). Older adults’ health informat...",LISR,NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [143]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [144]:
import os
for row, item in publications.iterrows():
    
    date_str = item.pub_date.strftime('%Y-%m-%d')  # Format the Timestamp as 'YYYY-MM-DD'
    md_filename = date_str + "-" + str(item.url_slug) + ".md"
    html_filename = date_str + "-" + str(item.url_slug)
    
    # md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    # html_filename = str(item.pub_date) + "-" + item.url_slug
    # year = item.pub_date.year
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""

    if len(str(item.type)) > 3:
        md += '\ncategory: "' + item.type + '"'
    else:
        md += 'category: "Other"'
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(date_str) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 

    if len(str(item.slides_url)) > 5:
        md += "| [Download slides here](" + item.slides_url + ")\n" 
     
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [145]:
!ls ../_publications/

2008-01-12-JKSIM.md  2017-01-07-eGEMs.md  2022-01-01-JASIST.md
2008-01-13-JKSLIS.md 2019-01-04-JASIST.md 2022-12-29-ACR.md
2015-01-08-MJLIS.md  2019-01-05-LISR.md   2022-12-30-Geron.md
2015-01-09-JKSLIS.md 2019-01-06-HIJ.md    2022-12-31-JAG.md
2015-01-10-LISR.md   2020-01-03-JAMIA.md  2023-12-27-AS.md
2015-01-11-JASIST.md 2021-01-02-IJMI.md   2023-12-28-IPM.md


In [146]:
!cat ../_publications/1980-10-21-paper-title-number-1.md 2017-05-19-paper-title-number-3.md

cat: ../_publications/1980-10-21-paper-title-number-1.md: No such file or directory
cat: 2017-05-19-paper-title-number-3.md: No such file or directory
